In [1]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=4.51", "accelerate", "sentence-transformers",
     "qdrant-client>=1.10,<2", "pandas", "scikit-learn", "python-dotenv", "tqdm",
     "vllm==0.25.1"],
    check=True,
)

# torchcodec la dependency tuy chon (audio/video) bi keo theo qua transformers/vllm;
# eager-probe cua no hay crash vi thieu libnvrtc.so.13 dung CUDA runtime, khong lien
# quan gi toi pipeline text-only o day -> go het truoc khi restart kernel.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchcodec"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
     "--no-deps", "typing_extensions>=4.13"],
    check=True,
)
# LUU Y: KHONG uninstall torchvision/torchaudio nua -- vLLM 0.25.1 import torchvision
# noi bo (vd. kernel_warmup cho model warmup path) du model dang dung khong lien quan
# anh/video; thieu no lam EngineCore crash ngay luc khoi dong (ModuleNotFoundError).

print('Cài đặt xong. QUAN TRỌNG: Restart Kernel rồi mới chạy các cell tiếp theo.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
modal 1.5.3 requires protobuf!=4.24.0,<7.0,>=3.19, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Cài đặt xong. QUAN TRỌNG: Restart Kernel rồi mới chạy các cell tiếp theo.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Tìm .env khi chạy tại root hoặc trong embedding/.
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path('.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.')

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
# ĐỔI MODEL: sửa đúng 1 dòng dưới đây (giữ nguyên đúng key trong MODEL_REPOS),
# rồi Restart Kernel + Run All. Không cần sửa gì khác để thử model kế tiếp.
AVAILABLE_MODELS = ['Qwen3.5-9B']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen3.5-9B': 'Qwen/Qwen3.5-9B',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    # Qwen3.5 official "Thinking mode for general tasks": temperature=1.0, top_p=0.95,
    # top_k=20, min_p=0.0, presence_penalty=1.5, repetition_penalty=1.0.
    # transformers.generate() KHÔNG có tham số presence_penalty (chỉ có ở API kiểu
    # OpenAI/vLLM) -> bỏ, không áp dụng qua generate() được. min_p có hỗ trợ native
    # từ transformers>=4.42 (notebook đang pin >=4.51 nên OK).
    'Qwen3.5-9B': {
        'profile_name': 'vendor_qwen3_5_thinking_general',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 1.0, 'top_p': 0.95,
            'top_k': 20, 'min_p': 0.0, 'repetition_penalty': 1.0,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v2'
BENCHMARK_PROTOCOL = 'vendor_recommended_multi_seed'
BGE_MODEL_ID = 'BAAI/bge-m3'
QDRANT_COLLECTION = 'laws_bge_m3_v2_correct_pooling'
EXPECTED_VECTOR_DIM = 1024
TOP_K = 14
MAX_INPUT_TOKENS = 24000
MAX_NEW_TOKENS = 8192
MAX_ARTICLE_CHARS = 6000  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # benchmark strict: không retry để chọn output hợp lệ hơn
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    # Modal Notebook: secret được attach lúc tạo notebook -> đã có sẵn trong os.environ,
    # không cần bước nào khác. Fallback Kaggle Secrets chỉ kích hoạt khi chạy trên Kaggle.
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
                if value:
                    return value
            except Exception:
                pass
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Thiếu secret, cần một trong: {names}')
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)
QDRANT_URL = get_secret('QDRANT_URL')
QDRANT_API_KEY = get_secret('QDRANT_API_KEY', 'QDRANT_KEY')

assert torch.cuda.is_available(), 'Notebook này yêu cầu GPU CUDA — khi tạo Modal Notebook nhớ chọn GPU (A10G trở lên cho model 7-8B).'
torch.manual_seed(EVAL_SEEDS[0])
torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('GPU:', torch.cuda.get_device_name(0))
print('Model:', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('Precision:', 'unquantized', 'BF16' if torch.cuda.is_bf16_supported() else 'FP16')


Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.
GPU: NVIDIA A100-SXM4-40GB
Model: Qwen3.5-9B -> Qwen/Qwen3.5-9B
Benchmark: v2 | protocol: vendor_recommended_multi_seed
Generation profile: vendor_qwen3_5_thinking_general | seeds: [2026]
Precision: unquantized BF16


In [2]:
def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PUBLIC_TEST_PATH=/path/to/file.json
    candidates = []
    if os.getenv('ALQAC_PUBLIC_TEST_PATH'):
        candidates.append(Path(os.environ['ALQAC_PUBLIC_TEST_PATH']))
    candidates += [
        # Modal Notebook: upload ALQAC2026_public_test.json qua file browser bên trái
        # (kéo thả vào đúng thư mục làm việc của notebook) -> sẽ khớp 1 trong 2 dòng dưới.
        Path('ALQAC2026_public_test.json'),
        Path('data/ALQAC2026_public_test.json'),
        Path('../data/ALQAC2026_public_test.json'),
        # Kaggle (giữ lại để notebook vẫn chạy được trên Kaggle nếu cần đối chiếu).
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh/ALQAC2026_public_test.json'),
        Path('/kaggle/working/ALQAC2026_public_test.json'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('ALQAC2026_public_test.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(
        f'Không tìm thấy ALQAC2026_public_test.json. Đã kiểm tra:\n{checked}\n'
        f'Trên Modal Notebook: upload file này qua file browser, hoặc set '
        f"os.environ['ALQAC_PUBLIC_TEST_PATH'] ở cell trước khi gọi find_public_test()."
    )

DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

assert len(public_data) == 50, f'Expected 50 cases, got {len(public_data)}'
assert len({x['case_id'] for x in public_data}) == len(public_data)
assert all(x.get('case_query') for x in public_data)
assert all(x.get('verdict_label') in LABELS for x in public_data)

# ---- Nạp CHUNK (agent_v4_results.json) làm case_facts bổ sung ngoài case_query ----
def find_data_file(name):
    candidates = []
    if os.getenv('ALQAC_' + name.upper().replace('.', '_') + '_PATH'):
        candidates.append(Path(os.environ['ALQAC_' + name.upper().replace('.', '_') + '_PATH']))
    candidates += [
        Path(name), Path('outputs') / name, Path('data') / name,
        Path('../outputs') / name, Path('../data') / name,
        Path('/kaggle/working') / name,
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(name))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f'Không tìm thấy {name}.')

AGENT_EVIDENCE_PATH = find_data_file('agent_v4_results.json')
with AGENT_EVIDENCE_PATH.open(encoding='utf-8') as f:
    agent_evidence_raw = json.load(f)

MAX_EVIDENCE_CHARS = int(os.getenv('ALQAC_MAX_EVIDENCE_CHARS', '8000'))

def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
    seen, parts, total = set(), [], 0
    for e in record.get('evidence_details', []):
        chunk_id = e.get('chunk_id')
        txt = str(e.get('text', '')).strip()
        if not txt or chunk_id in seen:
            continue
        seen.add(chunk_id)
        if total + len(txt) + 1 > max_chars:
            break
        parts.append(txt)
        total += len(txt) + 1
    return '\n'.join(parts)

evidence_by_case = {r['case_id']: _case_evidence_text(r) for r in agent_evidence_raw}


def _case_evidence_ids(record):
    """ID cac segment evidence cua case, GIU NGUYEN thu tu, bo trung.

    'case_evidence' trong submission la danh sach id segment (vd 'case_4101_seg_...').
    Uu tien khoa 'evidence' cua agent_v4; neu thieu thi lay chunk_id trong 'evidence_details'.
    """
    ids, seen = [], set()
    for value in (record.get('evidence') or []):
        sid = str(value).strip()
        if sid and sid not in seen:
            seen.add(sid)
            ids.append(sid)
    if not ids:
        for detail in (record.get('evidence_details') or []):
            sid = str(detail.get('chunk_id', '')).strip()
            if sid and sid not in seen:
                seen.add(sid)
                ids.append(sid)
    return ids


evidence_ids_by_case = {r['case_id']: _case_evidence_ids(r) for r in agent_evidence_raw}
_missing_ev = [x['case_id'] for x in public_data if not evidence_by_case.get(x['case_id'])]
if _missing_ev:
    print('CẢNH BÁO: thiếu evidence cho case:', _missing_ev)
print('Evidence bổ sung:', AGENT_EVIDENCE_PATH, '| phủ', len(public_data) - len(_missing_ev), '/', len(public_data), 'case')

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
inference_cases = [
    {'case_id': x['case_id'], 'case_query': x['case_query'],
     'case_facts': evidence_by_case.get(x['case_id'], '')}
    for x in public_data
]
gold_by_case = {x['case_id']: x['verdict_label'] for x in public_data}

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
if not qdrant.collection_exists(QDRANT_COLLECTION):
    raise RuntimeError(
        f'Chưa có collection {QDRANT_COLLECTION}. Hãy chạy notebook BGE-M3 trước.'
    )
collection_info = qdrant.get_collection(QDRANT_COLLECTION)
stored_dim = int(collection_info.config.params.vectors.size)
stored_count = int(qdrant.count(QDRANT_COLLECTION, exact=True).count)
assert stored_dim == EXPECTED_VECTOR_DIM, (stored_dim, EXPECTED_VECTOR_DIM)
assert stored_count == 3352, f'Expected 3352 laws, got {stored_count}'

embedder = SentenceTransformer(
    BGE_MODEL_ID, device='cuda' if torch.cuda.is_available() else 'cpu'
)
embedder.max_seq_length = 8192

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))
print('Qdrant:', QDRANT_COLLECTION, '| dim:', stored_dim, '| points:', stored_count)


Evidence bổ sung: /root/agent_v4_results.json | phủ 50 / 50 case


/usr/local/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.17.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dataset: /root/ALQAC2026_public_test.json
Cases: 50
Qdrant: laws_bge_m3_v2_correct_pooling | dim: 1024 | points: 3352


In [3]:
def retrieve_laws(case_query, case_facts='', top_k=TOP_K):
    # (2026-08-11) Embed CA case_query + case_facts (chunks agent_v4) -- thu nghiem
    # "full chunk" thay vi chi query trơn, khop thuc nghiem pre_retrieval/eval-retrieval-bge.ipynb.
    embed_text = (case_query.strip() + '\n\n' + case_facts.strip()).strip() if case_facts else case_query.strip()
    query_vector = embedder.encode(
        [embed_text or ' '], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    assert query_vector.shape == (EXPECTED_VECTOR_DIM,)
    assert np.isfinite(query_vector).all()
    hits = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    ).points
    laws = []
    seen = set()
    for rank, hit in enumerate(hits, 1):
        payload = hit.payload or {}
        key = (str(payload.get('law_id')), int(payload.get('aid')))
        if key in seen:
            continue
        seen.add(key)
        laws.append({
            'rank': rank,
            'score': float(hit.score),
            'law_id': key[0],
            'aid': key[1],
            'article_no': int(payload.get('article_no')),
            'content_Article': str(payload.get('content_Article') or ''),
        })
    assert len(laws) == top_k, f'Chỉ retrieve được {len(laws)}/{top_k} luật'
    return laws

sample_case = inference_cases[0]
sample_laws = retrieve_laws(sample_case['case_query'], sample_case.get('case_facts', ''))
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])

# Retrieve đủ 50 case trước, rồi giải phóng BGE-M3 để dành toàn bộ VRAM cho LLM full precision.
retrieval_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.json'
retrieval_meta_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.meta.json'
retrieval_signature_material = {
    'bge_model': BGE_MODEL_ID, 'collection': QDRANT_COLLECTION, 'top_k': TOP_K,
    'cases': inference_cases,
}
retrieval_signature = hashlib.sha256(
    json.dumps(retrieval_signature_material, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()
cached_signature = None
if retrieval_meta_path.exists():
    try:
        cached_signature = json.loads(retrieval_meta_path.read_text(encoding='utf-8')).get('signature')
    except Exception:
        pass
if retrieval_path.exists() and cached_signature == retrieval_signature:
    try:
        retrieval_cache = json.loads(retrieval_path.read_text(encoding='utf-8'))
    except Exception:
        retrieval_cache = {}
else:
    retrieval_cache = {}
for case in tqdm(inference_cases, desc='BGE-M3 retrieval'):
    if case['case_id'] not in retrieval_cache:
        retrieval_cache[case['case_id']] = retrieve_laws(case['case_query'], case.get('case_facts', ''))
        temp_path = retrieval_path.with_suffix('.json.tmp')
        temp_path.write_text(json.dumps(retrieval_cache, ensure_ascii=False, indent=2), encoding='utf-8')
        temp_path.replace(retrieval_path)
retrieval_meta_path.write_text(
    json.dumps({'signature': retrieval_signature, **retrieval_signature_material}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
expected_case_ids = {x['case_id'] for x in inference_cases}
assert set(retrieval_cache) == expected_case_ids
assert all(len(retrieval_cache[cid]) == TOP_K for cid in expected_case_ids)

del embedder
gc.collect()
torch.cuda.empty_cache()
print('Đã cache retrieval và giải phóng BGE-M3. GPU allocated:', round(torch.cuda.memory_allocated()/2**30, 2), 'GB')


,rank,score,law_id,article_no,aid
0,1,0.555824,91/2015/QH13,603,53373
1,2,0.532484,100/2015/QH13,266,56710
2,3,0.520990,91/2015/QH13,586,53356
3,4,0.516275,326/2016/UBTVQH14,27,13626
4,5,0.509900,100/2015/QH13,262,56706
5,6,0.508516,52/2014/QH13,74,53946
6,7,0.507149,100/2015/QH13,264,56708
7,8,0.507031,91/2015/QH13,595,53365
8,9,0.503605,93/2015/QH13,7,14312
9,10,0.502106,92/2015/QH13,359,51024


BGE-M3 retrieval:   0%|          | 0/50 [00:00<?, ?it/s]

Đã cache retrieval và giải phóng BGE-M3. GPU allocated: 0.01 GB


In [4]:
# Load model qua vLLM (batch nhiều case cùng lúc -> nhanh hơn transformers.generate() tuần tự).
# (2026-08-10) DOI TU AutoModelForCausalLM.generate() (tung case, khong batch) SANG vLLM
# batch -- giu NGUYEN prompt/schema/validate, chi doi co che sinh. (2026-08-11) fullchunks/bge-m3
# GIO CO case_facts (evidence agent_v4): retrieval embed ca case_query+case_facts (cell truoc),
# prompt cung dua case_facts vao qua facts_block -- moi ham o day nhan them tham so case_facts.
MODEL_DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# flashinfer JIT-compile kernel sampler co the FAIL neu moi truong thieu CUDA dev
# headers -> ep dung sampler PyTorch thuan, tranh crash EngineCore.
os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '0')

from vllm import LLM, SamplingParams

vllm_engine = LLM(
    model=MODEL_ID,
    dtype=MODEL_DTYPE,
    trust_remote_code=True,
    max_model_len=MAX_INPUT_TOKENS + MAX_NEW_TOKENS,
    max_num_seqs=64,
    gpu_memory_utilization=0.92,
    enforce_eager=True,
    seed=EVAL_SEEDS[0],
)
MODEL_REVISION = 'vllm-' + MODEL_ID
RESOLVED_GENERATION_CONFIG = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items()}
print('Loaded via vLLM:', MODEL_ID, '| dtype:', MODEL_DTYPE)
print('Resolved generation config:', json.dumps(RESOLVED_GENERATION_CONFIG, ensure_ascii=False))

SYSTEM_PROMPT = f'''Bạn là chuyên gia phân tích tranh chấp dân sự Việt Nam.
Bạn chỉ được sử dụng CASE_QUERY và {TOP_K} ĐIỀU LUẬT được cung cấp. Không được giả định dữ kiện ngoài đầu vào.
A là nguyên đơn, B là bị đơn. Hãy dự đoán đúng một trong bốn nhãn:
- A_WIN: toàn bộ hoặc về cơ bản toàn bộ yêu cầu của nguyên đơn được chấp nhận.
- B_WIN: yêu cầu của nguyên đơn bị bác toàn bộ hoặc về cơ bản toàn bộ.
- PARTIAL_A_WIN: nguyên đơn được chấp nhận một phần đáng kể nhưng không toàn bộ; kết quả nghiêng về A.
- PARTIAL_B_WIN: có phần yêu cầu của nguyên đơn được chấp nhận nhưng kết quả chủ yếu nghiêng về B.

Trả về đúng một JSON object, không Markdown, không văn bản bên ngoài JSON:
{{
  "prediction": "<LABEL>",
  "confidence": 0.78,
  "reasoning": "Lập luận ngắn gọn bằng tiếng Việt",
  "applied_laws": [
    {{"law_id": "91/2015/QH13", "aid": 53373, "reason": "Lý do áp dụng"}}
  ]
}}
Thay <LABEL> bằng đúng một trong A_WIN, B_WIN, PARTIAL_A_WIN, PARTIAL_B_WIN; không được giữ placeholder.
Chỉ chọn applied_laws từ danh sách {TOP_K} điều luật. Confidence phải nằm trong [0, 1].'''

BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'model_name': MODEL_NAME, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'generation_profile': ACTIVE_PROFILE,
    'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
    'dtype': str(MODEL_DTYPE), 'full_gpu_no_quantization': True,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_total_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': package_version('transformers'),
    'sentence_transformers': package_version('sentence-transformers'),
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases), 'labels': LABELS,
    'bge_model': BGE_MODEL_ID, 'qdrant_collection': QDRANT_COLLECTION, 'top_k_laws': TOP_K,
    'max_input_tokens': MAX_INPUT_TOKENS, 'max_new_tokens': MAX_NEW_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    'eval_seeds': EVAL_SEEDS,
    'invalid_output_policy': 'count_as_wrong',
    'system_prompt_sha256': hashlib.sha256(SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    'retrieval_signature': retrieval_signature,
}

def build_user_prompt(case_query, laws, case_facts=''):
    law_blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        law_blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f"{content}"
        )
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return (
        'CASE_QUERY:\n' + case_query.strip() + facts_block +
        f'\n\n{TOP_K} ĐIỀU LUẬT TRUY XUẤT:\n' + '\n\n'.join(law_blocks) +
        '\n\nHãy phân tích và trả về đúng JSON schema đã yêu cầu.'
    )

def build_messages(user_prompt):
    if ACTIVE_PROFILE['use_system_prompt']:
        return [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ]
    # DeepSeek-R1 khuyến nghị không dùng system role; nội dung hướng dẫn vẫn giữ nguyên.
    return [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user_prompt}]

def extract_first_json(text):
    text = re.sub(r'<think>.*?</think>', '', text or '', flags=re.I | re.S).strip()
    if '</think>' in text:
        text = text.rsplit('</think>', 1)[-1].strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.I | re.S).strip()
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    try:
        obj = ast.literal_eval(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    raise ValueError('Không tìm thấy JSON object hợp lệ')

def validate_prediction(obj, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'prediction không hợp lệ: {prediction!r}')
    confidence = float(obj.get('confidence'))
    if not 0.0 <= confidence <= 1.0:
        raise ValueError(f'confidence ngoài [0,1]: {confidence}')
    reasoning = str(obj.get('reasoning', '')).strip()
    if not reasoning:
        raise ValueError('reasoning rỗng')
    allowed = {(x['law_id'], int(x['aid'])) for x in retrieved_laws}
    clean_laws = []
    seen = set()
    for item in obj.get('applied_laws', []):
        try:
            key = (str(item['law_id']), int(item['aid']))
        except Exception:
            continue
        if key not in allowed or key in seen:
            continue
        seen.add(key)
        clean_laws.append({
            'law_id': key[0], 'aid': key[1],
            'reason': str(item.get('reason', '')).strip(),
        })
    return {
        'prediction': prediction,
        'confidence': confidence,
        'reasoning': reasoning,
        'applied_laws': clean_laws,
    }

class PredictionFormatError(RuntimeError):
    def __init__(self, message, raw_response, usage):
        super().__init__(message)
        self.raw_response = raw_response
        self.usage = usage

def _render_prompt(case_query, retrieved_laws, case_facts=''):
    user_prompt = build_user_prompt(case_query, retrieved_laws, case_facts)
    messages = build_messages(user_prompt)
    template_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        template_kwargs['enable_thinking'] = ACTIVE_PROFILE['enable_thinking']
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, **template_kwargs
    )
    input_tokens = len(tokenizer(rendered, truncation=False)['input_ids'])
    if input_tokens > MAX_INPUT_TOKENS:
        raise ValueError(f'Input vượt budget: {input_tokens}/{MAX_INPUT_TOKENS} tokens')
    return rendered, user_prompt, input_tokens

def _sampling_params_for(case_query, eval_seed):
    case_seed = eval_seed + int(hashlib.sha256(case_query.encode('utf-8')).hexdigest()[:8], 16)
    kwargs = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items() if k != 'do_sample'}
    return SamplingParams(max_tokens=MAX_NEW_TOKENS, seed=case_seed % (2**31 - 1), **kwargs), case_seed

def call_local_llm_batch(model_name, batch):
    """BATCH nhiều case trong 1 lệnh generate của vLLM (thay vòng lặp tuần tự cũ).
    batch = [(case_query, retrieved_laws, case_facts, eval_seed), ...]
    Trả list cùng thứ tự: (parsed_hoặc_None, raw_text, usage, user_prompt, error_hoặc_None)."""
    assert model_name == MODEL_NAME
    prompts, sampling_list, meta = [], [], []
    results = [None] * len(batch)
    for idx, (case_query, retrieved_laws, case_facts, eval_seed) in enumerate(batch):
        try:
            rendered, user_prompt, input_tokens = _render_prompt(case_query, retrieved_laws, case_facts)
        except Exception as exc:
            results[idx] = (None, None, None, None, exc)
            continue
        sp, case_seed = _sampling_params_for(case_query, eval_seed)
        prompts.append(rendered)
        sampling_list.append(sp)
        meta.append((idx, input_tokens, case_seed, user_prompt))

    if prompts:
        started = time.time()
        outputs = vllm_engine.generate(prompts, sampling_list)
        batch_duration = time.time() - started
        for (idx, input_tokens, case_seed, user_prompt), out in zip(meta, outputs):
            case_query, retrieved_laws, case_facts, eval_seed = batch[idx]
            gen = out.outputs[0]
            raw_text = (gen.text or '').strip()
            output_tokens = len(gen.token_ids)
            usage = {
                'input_tokens': input_tokens, 'output_tokens': output_tokens,
                'total_tokens': input_tokens + output_tokens,
                'hit_max_new_tokens': gen.finish_reason == 'length',
                'eval_seed': eval_seed, 'case_seed': case_seed,
                'batch_duration_seconds': round(batch_duration, 3),
            }
            if not raw_text:
                results[idx] = (None, raw_text, usage, user_prompt,
                                PredictionFormatError('Model trả output rỗng', raw_text, usage))
                continue
            try:
                parsed = validate_prediction(extract_first_json(raw_text), retrieved_laws)
                results[idx] = (parsed, raw_text, usage, user_prompt, None)
            except Exception as exc:
                results[idx] = (None, raw_text, usage, user_prompt,
                                PredictionFormatError(str(exc), raw_text, usage))
    return results

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

WARNING 08-11 14:49:21 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=644) INFO 08-11 14:49:38 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='Qwen/Qwen3.5-9B', speculative_config=None, tokenizer='Qwen/Qwen3.5-9B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Structur

(EngineCore pid=644) [transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=644) INFO 08-11 14:49:52 [gpu_model_runner.py:5209] Starting to load model Qwen/Qwen3.5-9B...
(EngineCore pid=644) INFO 08-11 14:49:52 [cuda.py:535] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=644) INFO 08-11 14:49:52 [mm_encoder_attention.py:373] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=644) INFO 08-11 14:49:52 [qwen_gdn_linear_attn.py:228] Using Triton/FLA GDN prefill kernel (requested=auto, head_k_dim=128).
(EngineCore pid=644) INFO 08-11 14:49:54 [cuda.py:476] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=644) INFO 08-11 14:49:54 [flash_attn.py:718] Using FlashAttention version 2
(EngineCore pid=644) INFO 08-11 14:51:33 [weight_utils.py:530] Time spent downloading weights for Qwen/Qwen3.5-9B: 97.898840 seconds
(EngineCore pid=644) INFO 08-11 14:51:33 [weight_utils.py:849] Filesystem type for checkpoin

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:02,  1.28it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.28it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.26it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.37it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.33it/s]
(EngineCore pid=644) 


(EngineCore pid=644) INFO 08-11 14:51:36 [default_loader.py:430] Loading weights took 3.02 seconds
(EngineCore pid=644) INFO 08-11 14:51:36 [gpu_model_runner.py:5306] Model loading took 17.66 GiB memory and 103.590452 seconds
(EngineCore pid=644) INFO 08-11 14:51:36 [interface.py:890] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
(EngineCore pid=644) INFO 08-11 14:51:36 [interface.py:914] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore pid=644) INFO 08-11 14:51:37 [gpu_model_runner.py:6322] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=644) INFO 08-11 14:53:24 [gpu_worker.py:538] Available KV cache memory: 16.66 GiB
(EngineCore pid=644) INFO 08-11 14:53:24 [kv_cache_utils.py:2146] GPU KV cache size: 520,102 tokens
(EngineCore pid=644) INFO 08-11 14:53:24 [kv_cache_uti

In [5]:
smoke_model = MODELS_TO_RUN[0]
smoke_seed = EVAL_SEEDS[0]
smoke_laws = retrieval_cache[sample_case['case_id']]
smoke_facts = sample_case.get('case_facts', '')
try:
    [(smoke_result, smoke_raw, smoke_usage, _smoke_prompt, smoke_err)] = call_local_llm_batch(
        smoke_model, [(sample_case['case_query'], smoke_laws, smoke_facts, smoke_seed)]
    )
    if smoke_err is not None:
        raise smoke_err
    print('Model:', smoke_model, '| seed:', smoke_seed)
    print(json.dumps(smoke_result, ensure_ascii=False, indent=2))
    print('Usage:', smoke_usage)
except Exception as exc:
    # Smoke lỗi không làm dừng benchmark; batch vẫn chấm case này đúng một lần theo seed.
    print('SMOKE WARNING:', repr(exc))

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=644) WARNING 08-11 14:54:32 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _zero_kv_blocks_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=644) WARNING 08-11 14:54:33 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _triton_mrope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=644) WARNING 08-11 14:54:34 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _causal_conv1d_update_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=644) WARNING 08-11 14:54:35 [jit_monitor.py:129] Triton kernel JIT compilation during inference: fused_recurrent_gated_delta_rule_packed_decode_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=644) WARNING 08-11 14:54:36 [jit_monitor.py:129] Triton kernel JIT c

Processed prompts: 100%|██████████| 1/1 [02:48<00:00, 168.45s/it, est. speed input: 63.86 toks/s, output: 21.22 toks/s]

Model: Qwen3.5-9B | seed: 2026
{
  "prediction": "PARTIAL_A_WIN",
  "confidence": 0.92,
  "reasoning": "Bị đơn là chủ sở hữu chó gây thiệt hại cho nguyên đơn, nên phải bồi thường theo Điều 603 BLDS 2015. Tuy nhiên, theo nội dung vụ việc, Tòa án chỉ chấp nhận một phần yêu cầu khởi kiện (tổng yêu cầu 6.875.000đ nhưng bị từ chối 3.437.500đ), do đó nguyên đơn thắng một phần đáng kể nhưng không toàn bộ.",
  "applied_laws": [
    {
      "law_id": "91/2015/QH13",
      "aid": 53373,
      "reason": "Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác."
    }
  ]
}
Usage: {'input_tokens': 10757, 'output_tokens': 3574, 'total_tokens': 14331, 'hit_max_new_tokens': False, 'eval_seed': 2026, 'case_seed': 743951647, 'batch_duration_seconds': 168.533}


In [6]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')

def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default

def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'generation_profile': ACTIVE_PROFILE,
        'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'], 'case_facts': case.get('case_facts', ''), 'laws': laws,
        'system_prompt': SYSTEM_PROMPT,
        'max_input_tokens': MAX_INPUT_TOKENS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    }
    raw = json.dumps(material, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

assert len(retrieval_cache) == len(inference_cases)

# all_model_results[model][seed][case_id] -> result
all_model_results = {}
for model in MODELS_TO_RUN:
    all_model_results[model] = {}
    for eval_seed in EVAL_SEEDS:
        result_path = OUTPUT_DIR / f'predictions_{slugify(model)}_seed-{eval_seed}.json'
        saved = load_json(result_path, {})
        print(f'\n=== {model} | seed={eval_seed} | cached {len(saved)}/{len(inference_cases)} ===')

        pending = []  # [(case, laws, cache_key), ...] - case CHUA co cache hop le
        for case in inference_cases:
            case_id = case['case_id']
            laws = retrieval_cache[case_id]
            cache_key = make_cache_key(model, case, laws, eval_seed)
            old = saved.get(case_id)
            # Cache cả output lỗi: không cho case thêm cơ hội chỉ vì lần trước sai format.
            if old and old.get('cache_key') == cache_key:
                continue
            pending.append((case, laws, cache_key))

        print(f'  Cần tính: {len(pending)}/{len(inference_cases)} case '
              f'(1 lệnh generate() batch qua vLLM thay vì {len(pending)} lệnh tuần tự)')

        if pending:
            batch_input = [
                (case['case_query'], laws, case.get('case_facts', ''), eval_seed)
                for case, laws, _cache_key in pending
            ]
            started = time.time()
            batch_results = call_local_llm_batch(model, batch_input)
            batch_duration = round(time.time() - started, 3)
            per_case_duration = round(batch_duration / max(1, len(pending)), 3)

            for (case, laws, cache_key), (parsed, raw_text, usage, _prompt, err) in zip(pending, batch_results):
                case_id = case['case_id']
                if err is None:
                    saved[case_id] = {
                        'case_id': case_id,
                        'case_query': case['case_query'],
                        'eval_seed': eval_seed,
                        **parsed,
                        'retrieved_laws': laws,
                        'raw_response': raw_text,
                        'usage': usage,
                        'duration_seconds': per_case_duration,
                        'generation_attempts': 1,
                        'cache_key': cache_key,
                        'error': None,
                    }
                else:
                    saved[case_id] = {
                        'case_id': case_id,
                        'case_query': case['case_query'],
                        'eval_seed': eval_seed,
                        'prediction': None,
                        'confidence': None,
                        'reasoning': '',
                        'applied_laws': [],
                        'retrieved_laws': laws,
                        'raw_response': getattr(err, 'raw_response', raw_text),
                        'usage': getattr(err, 'usage', usage),
                        'duration_seconds': per_case_duration,
                        'generation_attempts': 1,
                        'cache_key': cache_key,
                        'error': repr(err),
                    }
                    print(f'  INVALID {case_id} | seed={eval_seed}: {err}')
            atomic_write_json(result_path, saved)
            print(f'  Batch xong trong {batch_duration}s (~{per_case_duration}s/case).')
        all_model_results[model][eval_seed] = saved

print('Hoàn tất batch cho', len(EVAL_SEEDS), 'seed x', len(inference_cases), 'case.')

Benchmark manifest: outputs_alqac_e2e/v2/benchmark_manifest_qwen3-5-9b.json

=== Qwen3.5-9B | seed=2026 | cached 0/50 ===
  Cần tính: 50/50 case (1 lệnh generate() batch qua vLLM thay vì 50 lệnh tuần tự)


Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=644) WARNING 08-11 14:58:49 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 50/50 [09:53<00:00, 11.87s/it, est. speed input: 802.52 toks/s, output: 463.44 toks/s]

  INVALID case_4579 | seed=2026: prediction không hợp lệ: ''
  INVALID case_2238 | seed=2026: Không tìm thấy JSON object hợp lệ
  Batch xong trong 595.52s (~11.91s/case).
Hoàn tất batch cho 1 seed x 50 case.


In [7]:
def evaluate_run(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        prediction = item.get('prediction')
        is_valid_output = prediction in LABELS
        usage = item.get('usage') or {}
        rows.append({
            'model': model,
            'seed': eval_seed,
            'case_id': cid,
            'gold': gold_by_case[cid],
            'prediction': prediction,
            'scored_prediction': prediction if is_valid_output else INVALID_OUTPUT_LABEL,
            'is_valid_output': is_valid_output,
            'confidence': item.get('confidence'),
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'duration_seconds': item.get('duration_seconds'),
            'error': item.get('error'),
        })
    frame = pd.DataFrame(rows)
    valid = frame[frame['is_valid_output']].copy()
    n_total, n_valid = len(frame), len(valid)
    n_failed = n_total - n_valid
    if n_failed:
        failed_ids = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
        print(
            f'Cảnh báo {model} seed={eval_seed}: {n_failed}/{n_total} output không hợp lệ '
            f'được tính sai. Case: {failed_ids}'
        )

    scored_prediction = frame['scored_prediction']
    strict_correct = int((frame['gold'] == scored_prediction).sum())
    strict_accuracy = strict_correct / n_total if n_total else 0.0
    valid_accuracy = accuracy_score(valid['gold'], valid['prediction']) if n_valid else 0.0
    report = classification_report(
        frame['gold'], scored_prediction, labels=LABELS,
        output_dict=True, zero_division=0,
    ) if n_total else {}
    cm_all = confusion_matrix(
        frame['gold'], scored_prediction, labels=LABELS + [INVALID_OUTPUT_LABEL]
    ) if n_total else np.zeros((len(LABELS) + 1, len(LABELS) + 1), dtype=int)
    cm = cm_all[:len(LABELS), :]
    summary = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model,
        'seed': eval_seed,
        'n_total': n_total,
        'n_success': n_valid,
        'n_failed': n_failed,
        'invalid_output_rate': n_failed / n_total if n_total else 0.0,
        'coverage': n_valid / n_total if n_total else 0.0,
        'benchmark_valid': n_total == len(inference_cases),
        'all_outputs_valid': n_valid == n_total,
        'metric_scope': 'all_50_invalid_outputs_count_as_wrong',
        'strict_accuracy_all_50': strict_accuracy,
        'accuracy_successful_only': valid_accuracy,
        'macro_precision': report.get('macro avg', {}).get('precision', 0.0),
        'macro_recall': report.get('macro avg', {}).get('recall', 0.0),
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0.0),
        'weighted_f1': report.get('weighted avg', {}).get('f1-score', 0.0),
        'avg_input_tokens': frame['input_tokens'].mean(),
        'avg_output_tokens': frame['output_tokens'].mean(),
        'avg_duration_seconds': frame['duration_seconds'].mean(),
        'n_hit_max_new_tokens': int(frame['hit_max_new_tokens'].sum()),
    }
    per_label = pd.DataFrame([
        {
            'model': model,
            'seed': eval_seed,
            'label': label,
            'precision': report.get(label, {}).get('precision', 0.0),
            'recall': report.get(label, {}).get('recall', 0.0),
            'f1': report.get(label, {}).get('f1-score', 0.0),
            'support': int(report.get(label, {}).get('support', 0)),
        } for label in LABELS
    ])
    cm_frame = pd.DataFrame(
        cm,
        index=[f'gold_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS + [INVALID_OUTPUT_LABEL]],
    )
    return summary, per_label, cm_frame, frame

summaries = []
evaluation_artifacts = {}
for model, seed_results in all_model_results.items():
    evaluation_artifacts[model] = {}
    for eval_seed, results in seed_results.items():
        summary, per_label, cm_frame, case_frame = evaluate_run(model, eval_seed, results)
        summaries.append(summary)
        evaluation_artifacts[model][eval_seed] = {
            'per_label': per_label,
            'confusion_matrix': cm_frame,
            'cases': case_frame,
        }
        print(f'\n=== {model} | seed={eval_seed} ===')
        display(pd.DataFrame([summary]))
        display(cm_frame)

run_metrics = pd.DataFrame(summaries).sort_values(['model', 'seed']).reset_index(drop=True)
aggregate_metrics = run_metrics.groupby('model', as_index=False).agg(
    n_seeds=('seed', 'nunique'),
    accuracy_mean=('strict_accuracy_all_50', 'mean'),
    accuracy_std=('strict_accuracy_all_50', 'std'),
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    invalid_rate_mean=('invalid_output_rate', 'mean'),
    invalid_rate_std=('invalid_output_rate', 'std'),
    avg_output_tokens=('avg_output_tokens', 'mean'),
    avg_duration_seconds=('avg_duration_seconds', 'mean'),
)
aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']] = (
    aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']].fillna(0.0)
)
leaderboard = aggregate_metrics.sort_values(
    ['accuracy_mean', 'macro_f1_mean'], ascending=False
).reset_index(drop=True)

majority_label = pd.Series(list(gold_by_case.values())).value_counts().idxmax()
majority_accuracy = pd.Series(list(gold_by_case.values())).value_counts().max() / len(gold_by_case)
print(f'Majority baseline: {majority_label} | accuracy={majority_accuracy:.4f}')
print('Per-seed metrics:')
display(run_metrics)
print('Aggregate mean ± std across seeds:')
display(leaderboard)

for model, seed_artifacts in evaluation_artifacts.items():
    slug = slugify(model)
    model_runs = run_metrics[run_metrics['model'] == model]
    model_summary = leaderboard[leaderboard['model'] == model]
    model_runs.to_csv(
        OUTPUT_DIR / f'model_metrics_by_seed_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    model_summary.to_csv(
        OUTPUT_DIR / f'model_metrics_summary_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    for eval_seed, artifacts in seed_artifacts.items():
        suffix = f'{slug}_seed-{eval_seed}'
        artifacts['per_label'].to_csv(
            OUTPUT_DIR / f'metrics_per_label_{suffix}.csv', index=False, encoding='utf-8-sig'
        )
        artifacts['confusion_matrix'].to_csv(
            OUTPUT_DIR / f'confusion_matrix_{suffix}.csv', encoding='utf-8-sig'
        )
        artifacts['cases'].to_csv(
            OUTPUT_DIR / f'case_predictions_{suffix}.csv', index=False, encoding='utf-8-sig'
        )


Cảnh báo Qwen3.5-9B seed=2026: 2/50 output không hợp lệ được tính sai. Case: ['case_4579', 'case_2238']

=== Qwen3.5-9B | seed=2026 ===


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,Qwen3.5-9B,2026,50,48,2,0.04,0.96,True,...,0.58,0.604167,0.443492,0.50148,0.457385,0.539506,9524.74,5500.36,11.91,2


,pred_A_WIN,pred_B_WIN,pred_PARTIAL_A_WIN,pred_PARTIAL_B_WIN,pred___INVALID_OUTPUT__
gold_A_WIN,15,0,1,0,0
gold_B_WIN,1,7,1,0,1
gold_PARTIAL_A_WIN,7,5,7,0,0
gold_PARTIAL_B_WIN,0,1,3,0,1


Majority baseline: PARTIAL_A_WIN | accuracy=0.3800
Per-seed metrics:


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,Qwen3.5-9B,2026,50,48,2,0.04,0.96,True,...,0.58,0.604167,0.443492,0.50148,0.457385,0.539506,9524.74,5500.36,11.91,2


Aggregate mean ± std across seeds:


,model,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,invalid_rate_std,avg_output_tokens,avg_duration_seconds
0,Qwen3.5-9B,1,0.58,0.0,0.457385,0.0,0.04,0.0,5500.36,11.91


In [8]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                # case_evidence = id cac segment evidence THAT cua case (agent_v4),
                # khop cau truc voi pado/private -- do duoc Case Recall cong bang.
                'case_evidence': evidence_ids_by_case.get(case['case_id'], []),
                # law_evidence: retrieval THAT co xay ra qua Qdrant (retrieval_cache) nhung
                # applied_laws thuong thieu/rong (model khong dien du) -> lay TOAN BO 10 dieu
                # da retrieval, khop cach lam voi pado/fullchunks-base.
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in retrieval_cache.get(case['case_id'], [])
                ],
            })
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| evaluated:', len(inference_cases), '/ 50',
            '| invalid counted wrong:', n_failed,
            '| valid submission rows:', len(submission), '/ 50',
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


Qwen3.5-9B | seed: 2026 | evaluated: 50 / 50 | invalid counted wrong: 2 | valid submission rows: 48 / 50 | outputs_alqac_e2e/v2/submission_qwen3-5-9b_seed-2026.json
Outputs: /root/outputs_alqac_e2e/v2
